# Connecting Strands Agents with AWS services (Working Version)

## Overview
This notebook demonstrates how to connect Strands Agents to AWS services without requiring SSM permissions.

**Features:**
- Uses environment variables instead of SSM Parameter Store
- Unique resource names (customize with your name)
- Automatic DynamoDB table creation
- Comprehensive AWS credential validation

In [ ]:
# Install requirements
!pip install strands-agents strands-agents-tools boto3 pandas

In [ ]:
import os
import boto3
from strands import Agent, tool
from strands.models import BedrockModel
from botocore.exceptions import ClientError, NoCredentialsError

In [ ]:
# Test AWS credentials
try:
    sts = boto3.client('sts')
    account_info = sts.get_caller_identity()
    print(f"✅ Connected to AWS account: {account_info['Account']}")
except Exception as e:
    print(f"❌ AWS connection failed: {e}")
    print("Please configure AWS credentials before continuing.")

In [ ]:
# Configuration with unique names - CHANGE 'YourName' to your actual name!
table_name = 'restaurant-bookings-YourName'
kb_id = 'kb-id-YourName'

dynamodb = boto3.resource('dynamodb')
print(f"DynamoDB table: {table_name}")
print(f"Knowledge Base ID: {kb_id}")

In [ ]:
# Create DynamoDB table if needed
try:
    table = dynamodb.Table(table_name)
    table.load()
    print(f"✅ Table {table_name} exists")
except dynamodb.meta.client.exceptions.ResourceNotFoundException:
    print(f"Creating table {table_name}...")
    table = dynamodb.create_table(
        TableName=table_name,
        KeySchema=[
            {'AttributeName': 'booking_id', 'KeyType': 'HASH'},
            {'AttributeName': 'restaurant_name', 'KeyType': 'RANGE'}
        ],
        AttributeDefinitions=[
            {'AttributeName': 'booking_id', 'AttributeType': 'S'},
            {'AttributeName': 'restaurant_name', 'AttributeType': 'S'}
        ],
        BillingMode='PAY_PER_REQUEST'
    )
    table.wait_until_exists()
    print(f"✅ Table {table_name} created")

In [ ]:
@tool
def get_booking_details(booking_id: str, restaurant_name: str) -> dict:
    """Get booking details by ID and restaurant name"""
    try:
        response = table.get_item(
            Key={'booking_id': booking_id, 'restaurant_name': restaurant_name}
        )
        if 'Item' in response:
            return response['Item']
        else:
            return f"No booking found with ID {booking_id}"
    except Exception as e:
        return str(e)

In [ ]:
@tool
def create_booking(date: str, hour: str, restaurant_name: str, guest_name: str, num_guests: int) -> str:
    """Create a new restaurant booking"""
    import uuid
    try:
        booking_id = str(uuid.uuid4())[:8]
        table.put_item(
            Item={
                'booking_id': booking_id,
                'restaurant_name': restaurant_name,
                'date': date,
                'name': guest_name,
                'hour': hour,
                'num_guests': num_guests
            }
        )
        return f"Reservation created with booking ID: {booking_id}"
    except Exception as e:
        return str(e)

In [ ]:
@tool
def delete_booking(booking_id: str, restaurant_name: str) -> str:
    """Delete an existing booking"""
    try:
        response = table.delete_item(
            Key={'booking_id': booking_id, 'restaurant_name': restaurant_name}
        )
        if response['ResponseMetadata']['HTTPStatusCode'] == 200:
            return f'Booking with ID {booking_id} deleted successfully'
        else:
            return f'Failed to delete booking with ID {booking_id}'
    except Exception as e:
        return str(e)

In [ ]:
# Setup model
MODEL_ID = 'anthropic.claude-3-5-sonnet-20241022-v2:0'
model = BedrockModel(model_id=MODEL_ID)
print(f"Using model: {MODEL_ID}")

In [ ]:
# System prompt
system_prompt = """You are Restaurant Helper, a restaurant assistant helping customers with reservations. 
You can create bookings, get booking details, or delete reservations. Always be polite and mention your name.
If you cannot help with something, provide this phone number: +1 999 999 99 9999."""

In [ ]:
# Create agent
from strands_tools import current_time

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[current_time, get_booking_details, create_booking, delete_booking]
)

print("✅ Agent created successfully!")

In [ ]:
# Test the agent
response = agent("Hi! Can you help me make a reservation?")
print(response)

In [ ]:
# Make a test booking
response = agent("Make a reservation for 2 people at Rice & Spice for tomorrow at 7pm under the name John")
print(response)

In [ ]:
# Check table contents
import pandas as pd

def get_all_bookings():
    response = table.scan()
    items = response['Items']
    while 'LastEvaluatedKey' in response:
        response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
        items.extend(response['Items'])
    return pd.DataFrame(items)

bookings = get_all_bookings()
print("Current bookings:")
bookings